# Rigid molecular assembly with ASE

This notebook demonstrates the reusable `ase-molecular-assembly` modules. Every monomer is treated as a rigid body. Plane atoms and axis atoms are selected explicitly; the workflow does not infer π-cores, end groups, or intermolecular chemistry from an XYZ file. Distances are in Å and angles are in degrees.

## A. Setup

The path setup works when the notebook is launched from this notebook folder, the project folder, or the repository root. GUI viewing is disabled by default so that the complete notebook can run headlessly.

In [ ]:
from pathlib import Path
import sys

import numpy as np
from ase.build import molecule
from ase.io import read, write
from ase.visualize import view

working_directory = Path.cwd().resolve()
candidates = [
    working_directory,
    working_directory.parent,
    working_directory / "ase-molecular-assembly",
]
PROJECT_DIR = next(
    path for path in candidates
    if (path / "frame.py").is_file() and (path / "assembly.py").is_file()
)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from assembly import (
    Component,
    LateralPlacement,
    VerticalInterface,
    add_lateral_components,
    build_horizontal_assembly,
    build_vertical_stack,
)
from frame import define_frame
from validation import validate_assembly

OUTPUT_DIR = PROJECT_DIR / "notebook" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ENABLE_GUI = False
print(f"Project: {PROJECT_DIR}")
print(f"Local output: {OUTPUT_DIR}")

## B. Load or create monomers

ASE's built-in benzene geometry gives a self-contained planar example. `monomer_b` changes one atom label only to exercise an ordered heterogeneous assembly; it is a geometry demonstration, not a claim of an optimised molecular structure. For real data, replace the construction with `monomer_a = read("monomer.xyz")` and choose scientifically justified plane and axis atoms.

In [ ]:
monomer_a = molecule("C6H6")
monomer_b = monomer_a.copy()
monomer_b[0].symbol = "N"

# Real-data replacement:
# monomer_a = read("monomer.xyz")

print("A:", monomer_a.get_chemical_formula(), len(monomer_a), "atoms")
print("B:", monomer_b.get_chemical_formula(), len(monomer_b), "atoms")

## C. Define a local molecular frame

The six ring atoms define a best-fit plane by SVD. The ordered pair `(0, 1)` fixes positive local x after projection into that plane. The explicit normal reference fixes the normal sign. Local y is constructed so that `cross(x, y) = normal`.

In [ ]:
plane_indices = list(range(6))
x_axis_indices = (0, 1)
normal_reference = (0.0, 0.0, 1.0)

frame_a = define_frame(
    monomer_a, plane_indices, x_axis_indices, normal_reference=normal_reference
)
frame_b = define_frame(
    monomer_b, plane_indices, x_axis_indices, normal_reference=normal_reference
)
component_a = Component(monomer_a, frame_a, label="A", source="ASE C6H6 demo")
component_b = Component(monomer_b, frame_b, label="B", source="label-modified demo")

print("origin:", frame_a.origin)
print("x axis:", frame_a.x_axis)
print("y axis:", frame_a.y_axis)
print("normal:", frame_a.normal)
print("basis.T @ basis:\n", np.round(frame_a.basis.T @ frame_a.basis, 12))
print("right handed:", np.allclose(np.cross(frame_a.x_axis, frame_a.y_axis), frame_a.normal))

## D. Vertical assembly

For $N$ ordered components, supply exactly $N-1$ interfaces. Interface 0 places component 1 relative to component 0; interface 1 places component 2 relative to component 1. Separations and slips accumulate in the first component's local frame, and each twist is relative to the preceding component.

In [ ]:
dimer_interface = VerticalInterface(
    normal_separation=3.4, slip_x=0.8, slip_y=0.2, twist_degrees=18.0
)
aa_dimer = build_vertical_stack([component_a, component_a], [dimer_interface])

trimer_interfaces = [
    VerticalInterface(3.4, slip_x=0.5, twist_degrees=12.0),
    VerticalInterface(3.6, slip_x=-0.2, slip_y=0.3, twist_degrees=-8.0),
]
aaa_trimer = build_vertical_stack(
    [component_a, component_a, component_a], trimer_interfaces
)
ab_dimer = build_vertical_stack(
    [component_a, component_b],
    [VerticalInterface(3.5, slip_y=0.6, twist_degrees=25.0)],
)

print("A-A dimer:", len(aa_dimer), "atoms")
print("A-A-A trimer:", len(aaa_trimer), "atoms")
print("A-B order:", [item["label"] for item in ab_dimer.info["assembly"]["components"]])

## E. Horizontal assembly

Each guest frame is aligned with the host frame, rotated about the host normal through the common origin, and then translated by `(local x, local y, local normal)`. These are explicit placements; no donor, acceptor, or end-group recognition is attempted.

In [ ]:
lateral_dimer = build_horizontal_assembly(
    component_a,
    [LateralPlacement(component_b, translation_local=(7.0, 0.0, 0.0), rotation_degrees=30.0)],
)
lateral_cluster = build_horizontal_assembly(
    component_a,
    [
        LateralPlacement(component_a, (7.0, 0.0, 0.0), 30.0),
        LateralPlacement(component_b, (-7.0, 1.5, 0.0), -40.0),
        LateralPlacement(component_a, (0.0, 7.0, 0.0), 90.0),
    ],
)
print("lateral dimer component IDs:", np.unique(lateral_dimer.arrays["component_id"]))
print("multi-partner component IDs:", np.unique(lateral_cluster.arrays["component_id"]))

## F. Mixed assembly

Mixed motifs are compositions of the same primitives. Here a lateral B component is added to the A-A-A vertical stack without a separate motif-specific algorithm. Existing component IDs remain unchanged and the new component receives the next ID.

In [ ]:
mixed_assembly = add_lateral_components(
    aaa_trimer,
    frame_a,
    [LateralPlacement(component_b, (7.0, 0.0, 0.0), 60.0)],
)
print("mixed type:", mixed_assembly.info["assembly"]["assembly_type"])
print("mixed component IDs:", np.unique(mixed_assembly.arrays["component_id"]))

## G. Validation

Validation compares atom counts, component compositions, and intra-component pairwise distances with the supplied original monomers. It also reports the closest inter-component contact and treats contacts below the configured clash threshold as failures. Vertical requests can additionally be checked against recovered component frames.

In [ ]:
trimer_report = validate_assembly(
    aaa_trimer,
    [monomer_a, monomer_a, monomer_a],
    reference_frames=[frame_a, frame_a, frame_a],
    vertical_interfaces=trimer_interfaces,
)
mixed_report = validate_assembly(
    mixed_assembly, [monomer_a, monomer_a, monomer_a, monomer_b]
)
assert trimer_report.passed, trimer_report.errors
assert mixed_report.passed, mixed_report.errors
print("trimer status:", trimer_report.status)
print("maximum rigid-body error / Å:", trimer_report.maximum_rigid_body_error)
print("minimum mixed inter-component distance / Å:", mixed_report.minimum_intercomponent_distance)
trimer_report.to_dict()

## H. Visualisation

Set `ENABLE_GUI = True` in an interactive session to request ASE's viewer. The default path remains headless and therefore suitable for automated execution.

In [ ]:
if ENABLE_GUI:
    view(mixed_assembly)
else:
    print("GUI display skipped; set ENABLE_GUI = True for an interactive ASE viewer.")

## I. Export

Plain XYZ stores elements and coordinates. Extended XYZ also stores the per-atom `component_id` array, making it the more useful choice when fragment identity must survive a round trip. Both files remain in the notebook's ignored local `output` directory.

In [ ]:
xyz_path = OUTPUT_DIR / "mixed_assembly.xyz"
extxyz_path = OUTPUT_DIR / "mixed_assembly.extxyz"
write(xyz_path, mixed_assembly, format="xyz")
write(extxyz_path, mixed_assembly, format="extxyz")
round_trip = read(extxyz_path)
assert np.array_equal(round_trip.arrays["component_id"], mixed_assembly.arrays["component_id"])
print(xyz_path)
print(extxyz_path)
print("extended-XYZ component IDs preserved:", np.unique(round_trip.arrays["component_id"]))